In [7]:
import pandas as pd
import numpy as np
import plotly as plt
import plotly.graph_objects as go
import sys
sys.path.append('../import')
import rdkit
#print (rdkit.__version__)
#print (rdkit.__file__)
#import drop_compounds

CHEMBL_with_pka_db=pd.read_csv("CHEMBL_with_pka_database_with_descriptors.csv",encoding='unicode_escape')
PUBCHEM_with_pka_db=pd.read_csv("PubChem_with_pka_database_with_descriptors.csv",encoding='unicode_escape')
GPKA_db=pd.read_csv("Gpka_database_with_descriptors.csv")
SAMPL_db=pd.read_csv("SAMPL_database_with_descriptors.csv")

#remove sampl7
SAMPL_db=SAMPL_db[~SAMPL_db["name"].str.startswith("sm7")]

for db in [CHEMBL_with_pka_db,PUBCHEM_with_pka_db,GPKA_db,SAMPL_db]: 
    db.set_index("name",inplace=True)
    db.dropna(how='all', axis=1, inplace=True)
    db["PSA/MW"]=db["PSA"]/db["MW"]
#for d in drop_compounds.drop_compounds:    GPKA_db =GPKA_db[GPKA_db["compn"].str.startswith(d)==False]

CHEMBL_with_pka_small_db= CHEMBL_with_pka_db[CHEMBL_with_pka_db["MW"]<250]


In [8]:

import scipy

properties=["MW","n_rot_bonds","ALOGP","PSA","PSA/MW"]
name_of_properties=["Molecuar Weight","number of rotatable bonds","Calculated LOGP","Topological polar surface area",
                   "(Topological polar surface area)/MW"]
x_scales=[[0,800],[-1,16],[0,300],[-5,10],[-0.05,0.05]]
n_bins=[400,400,300,200,200]
name_of_files=["Molecuar_Weight_histogram","number_of_rotatable_bonds_histogram",
               "Calculated_LOGP_histogram","Topological_polar_surface_area_histogram",
               "Topological_polar_surface_area_histogram_relative" ]

for prop,name_of_property,x_scale,n_bins,file_name in zip(properties,name_of_properties,x_scales,n_bins,name_of_files):
    
    MW_histograms=[]
    scatters=[]
    for db,color,name in zip([GPKA_db,SAMPL_db,CHEMBL_with_pka_db,PUBCHEM_with_pka_db],
                             ["red","orange","green","blue"],["Gpka","SAMPL","CHEMBL(pKa)","PUBCHEM(pKa)"]):
        #MW_histograms.append(go.Histogram(x=db["MW"],opacity=0.75,xbins={"size":0.10},marker_color=color,legendgroup=1,legendrank=2))
        
        db[prop]=np.nan_to_num(db[prop])
        histogram=go.Histogram(x=list(db[prop]),opacity=1.0,marker_color=color,histnorm='percent',name=name,#nbinsx=n_bins,
                                autobinx=False,
                                xbins=dict(start=x_scale[0],end=x_scale[1], size=(x_scale[1]-x_scale[0])/20)
                              )
        xx=np.linspace(histogram.xbins['start'],histogram.xbins['end']+histogram.xbins['size'],100)
              
        try: 
            density = scipy.stats.gaussian_kde(db[prop])
            yy=density(xx)
            #calculate the scale
            plotbins = list(np.arange(start=histogram.xbins['start'], stop=histogram.xbins['end']+histogram.xbins['size'], step=histogram.xbins['size']))
            counts, bins = np.histogram(db[prop], bins=plotbins)
            scale_factor=100*np.max(counts)/(np.max(yy)*np.sum(counts))
            yy=yy*scale_factor
        except: 
             print ("fail")
             yy=np.zeros(len(xx))
 
        
        sct=go.Scatter(x=list(xx),y=list(yy),mode='lines',showlegend=False,hoverinfo="skip",
                       line_color=color,
                       #marker=dict(color=color, line=dict(width=1),showscale=False),
                       fill='tozeroy',
                      )
        #go.Figure(sct).show()
        MW_histograms.append(histogram)
        scatters.append(sct)
    
    fig=go.Figure(data=MW_histograms[0])
    fig.add_trace(scatters[0])
    fig.add_trace(MW_histograms[1])
    fig.add_trace(scatters[1])
    fig.add_trace(MW_histograms[2])
    fig.add_trace(scatters[2])
    fig.add_trace(MW_histograms[3])
    fig.add_trace(scatters[3])
  

    fig.update_layout(height=800,xaxis_range=x_scale,#yaxis_range=[0,6],
                      legend={"yanchor":"top","xanchor":"right","y":0.95,"x":0.92,"font":{"size":28}},
                        bargap=0.4,bargroupgap=0)
    fig.update_xaxes(title_text=name_of_property,title_font={'size': 28, 'weight': 1000},tickfont={"size":24})
    fig.update_yaxes(title_text="%",title_font={'size': 28, 'weight': 1000},tickfont={"size":24})
    
    #fig.show()
    fig.write_html(file_name+".html")
    fig.write_image(file_name+".png", width=1200, height=800,scale=1)




In [9]:
from collections import Counter
from db_analysis_functions import shannon_entropy_scaffolds
from db_analysis_functions import normalized_shannon_entropy_scaffolds



GPKA_Murcko_Scaffolds_counts={}
CHEMBL_Murcko_Scaffolds_counts={}
PUBCHEM_Murcko_Scaffolds_counts={}
SAMPL_Murcko_scaffolds_counts={}
"""
reference_Murcko_Scaffolds_counts=Counter(list(GPKA_db["Murcko_Scaffold"])+
                                    list(CHEMBL_with_pka_db["Murcko_Scaffold"])+
                                    list(PUBCHEM_with_pka_db["Murcko_Scaffold"])+
                                    list(SAMPL_db["Murcko_Scaffold"]))
"""
ref_scaffolds=[]
for db in [PUBCHEM_with_pka_db,CHEMBL_with_pka_db]:   
    reference_Murcko_Scaffolds_counts=Counter(list(db["Murcko_Scaffold"]))
    reference_Murcko_Scaffolds_counts=dict(reference_Murcko_Scaffolds_counts.most_common())
    ref_scaffolds.append(reference_Murcko_Scaffolds_counts)

flag=True
       
for ref_scaffold,ref_name in zip(ref_scaffolds,["PUBCHEM","CHEMBL"]):                                   
    
    Murcko_histograms=[]
    for db,name,db_Murcko_scf_counts,color in zip([GPKA_db,SAMPL_db,CHEMBL_with_pka_db,PUBCHEM_with_pka_db],
                            ["GPKA              ","SAMPL            ","CHEMBL(pKa)  ","PUBCHEM(pKa)"],
                            [GPKA_Murcko_Scaffolds_counts,SAMPL_Murcko_scaffolds_counts,CHEMBL_Murcko_Scaffolds_counts,PUBCHEM_Murcko_Scaffolds_counts],
                            ["red","orange","green","blue"]):
        #Murcko_scf_counts={}
        Murcko_scf_counts=[]
        for k in ref_scaffold.keys():
            db_Murcko_scf_counts[k]=list(db["Murcko_Scaffold"]).count(k)/len(db)
            Murcko_scf_counts.append(db_Murcko_scf_counts[k])

        if flag:
            shannon_entropy=normalized_shannon_entropy_scaffolds(db["Murcko_Scaffold"])
            if shannon_entropy>=10:
                name_in_legend= name +(" scld. Shannon entropy: "+"{:.2f}".format(shannon_entropy)).rjust(10," ")
            else:    
                name_in_legend= name +(" scld. Shannon entropy: "+"{:.3f}".format(shannon_entropy)).rjust(10," ")
        else: name_in_legend=name
        x_axis_size=np.max([ len(str(t)) for t in list(ref_scaffold.keys())[0:20]  ])
        histogram=go.Bar(x=list(ref_scaffold.keys())[0:20], #list(range(0,len(Murcko_scf_counts)))[0:20],
                                        y=Murcko_scf_counts[0:20],
                                        opacity=1.0,
                                        marker_color=color,
                                        name=name_in_legend,
                                        #text= list(all_Murcko_Scaffolds_counts.keys())[0:20],hoverinfo="text"
                        )
    
            
        Murcko_histograms.append(histogram)
    
    
        
    fig2=go.Figure(data=Murcko_histograms[0])
    fig2.add_trace(Murcko_histograms[1])
    fig2.add_trace(Murcko_histograms[2])
    fig2.add_trace(Murcko_histograms[3])
    print(x_axis_size)
    #x_axis_size=0
    
    fig2.update_layout(height=800+8*x_axis_size,width=1100,#xaxis_range=x_scale,#yaxis_range=[0,6],
                        autosize=False,
                           margin=dict(
                                        l=50,
                                        r=50,
                                        b=8*x_axis_size,
                                        t=50,
                                        pad=4
                                    ),
                        #minreducedwidth=1100,
                        #minreducedheight=800,
                      legend={"yanchor":"top","xanchor":"right","y":0.85,"x":0.9,"font":{"size":28}})
    fig2.update_xaxes(title_text="",title_font={'size': 28, 'weight': 1000},tickfont={"size":16},tickangle = 60)
    fig2.update_yaxes(title_text="",title_font={'size': 28, 'weight': 1000},tickfont={"size":24})
    fig2.update_xaxes(automargin=True)
    #fig2.show()
    fig2.write_html("most_fequent_Murcko_Scaffolds_wr_"+ref_name+".html")
    fig2.write_image("most_fequent_Murcko_Scaffolds"+ref_name+".png", width=1600, height=1600,scale=4)
    
    print ("20 most frequent scaffolds in CHEMBL database")
    for s in list(ref_scaffold.keys())[0:20]: print(s)
    flag=False



21
20 most frequent scaffolds in CHEMBL database
c1ccccc1
nan
c1ccncc1
c1cncnc1
c1ccc2ccccc2c1
c1ccc2ncccc2c1
N=c1nccc[nH]1
N=c1ccnc[nH]1
c1c[nH]cn1
c1ncc2nc[nH]c2n1
C1CCNCC1
c1ccc2ncncc2c1
c1cnc2ncncc2n1
C1CCCCC1
c1cn[nH]c1
c1ccc2[nH]ccc2c1
c1ccc2[nH]cnc2c1
c1ccc(Cc2ccccc2)cc1
c1ccc(N=Nc2ccccc2)cc1
c1cc[nH+]cc1
44
20 most frequent scaffolds in CHEMBL database
c1ccccc1
nan
c1ccc(N=c2c3ccccc3[nH]c3ccccc23)cc1
c1ccncc1
c1ccc(Nc2c3ccccc3nc3ccccc23)cc1
c1ccc2ncccc2c1
c1ccc(NC2=NCCN2)cc1
[CH]=C1NC=CN1
c1ccc2nc3ccccc3cc2c1
O=S(=O)(Nc1ccccc1)c1ccccc1
c1c[nH]cn1
O=C1C2C(C(=O)N1Cc1ccccc1)C(c1ccccc1)N1CCCC21
c1ccc(N2CCNCC2)cc1
c1ccc(Oc2ccccc2)cc1
c1ccc(C2CCc3ccc4[nH]cnc4c3O2)cc1
c1ccc(-c2ccccc2)cc1
O=c1ccc2ccccc2o1
O=c1cc[nH]cc1
N=C1COCC(c2cccc(NC(=O)c3ccccn3)c2)N1
C(=Cc1ccccc1)C=Nc1ccccc1


In [10]:
CHEMBL_with_pka_db=pd.read_csv("CHEMBL_with_pka_database_with_descriptors_tanimoto_with_Gpka.csv",encoding='unicode_escape')
#CHEMBL_with_pka_small_db= CHEMBL_with_pka_db[CHEMBL_with_pka_db["MW"]<300]
PUBCHEM_with_pka_db=pd.read_csv("PubChem_with_pka_database_with_descriptors_tanimoto_with_Gpka.csv",encoding='unicode_escape')
GPKA_db=pd.read_csv("Gpka_database_with_descriptors_tanimoto_with_Gpka.csv")
SAMPL_db=pd.read_csv("SAMPL_database_with_descriptors_tanimoto_with_Gpka.csv")
import scipy.stats

    
for db in [CHEMBL_with_pka_db,SAMPL_db,PUBCHEM_with_pka_db,GPKA_db]: 
    db.set_index("name",inplace=True)
    db.dropna(how='all', axis=1, inplace=True)


props=["Morgan radius 2 tanimoto similarity average","Morgan radius 2 tanimoto similarity max",
      "Morgan radius 2 tanimoto similarity median","Morgan radius 2 tanimoto similarity percentile 80",
      "Morgan radius 2 tanimoto similarity percentile 90","Morgan radius 2 tanimoto similarity percentile 90",
        "Morgan radius 3 tanimoto similarity average","Morgan radius 3 tanimoto similarity max",
      "Morgan radius 3 tanimoto similarity median","Morgan radius 3 tanimoto similarity percentile 80",
      "Morgan radius 3 tanimoto similarity percentile 90","Morgan radius 3 tanimoto similarity percentile 90",      
        "Morgan radius 4 tanimoto similarity average","Morgan radius 4 tanimoto similarity max",
      "Morgan radius 4 tanimoto similarity median","Morgan radius 4 tanimoto similarity percentile 80",
      "Morgan radius 4 tanimoto similarity percentile 90","Morgan radius 4 tanimoto similarity percentile 90", 
     "custom fingerprint tanimoto similarity average","custom fingerprint tanimoto similarity max",
      "custom fingerprint tanimoto similarity median","custom fingerprint tanimoto similarity percentile 80",
      "custom fingerprint tanimoto similarity percentile 90","custom fingerprint tanimoto similarity percentile 90",      
      ]
names_of_prop=["Morgan (R=2) fingerprint Tanimoto index (average)","Morgan (R=2) fingerprint Tanimoto index (max)",
      "Morgan (R=2) fingerprint Tanimoto index (median)","Morgan (R=2) fingerprint Tanimoto index (percentile 80)",
        "Morgan (R=2) fingerprint Tanimoto index (percentile 90)","Morgan (R=2) fingerprint Tanimoto index (percentile 95)",
       "Morgan (R=3) fingerprint Tanimoto index (average)","Morgan (R=3) fingerprint Tanimoto index (max)",
      "Morgan (R=3) fingerprint Tanimoto index (median)","Morgan (R=3) fingerprint Tanimoto index (percentile 80)",
        "Morgan (R=3) fingerprint Tanimoto index (percentile 90)","Morgan (R=3) fingerprint Tanimoto index (percentile 95)",
       "Morgan (R=4) fingerprint Tanimoto index (average)","Morgan (R=4) fingerprint Tanimoto index (max)",
      "Morgan (R=4) fingerprint Tanimoto index (median)","Morgan (R=4) fingerprint Tanimoto index (percentile 80)",
        "Morgan (R=4) fingerprint Tanimoto index (percentile 90)","Morgan (R=4) fingerprint Tanimoto index (percentile 95)",
       "functional groups fingerprint Tanimoto index (average)","functional groups fingerprint Tanimoto index (max)",
      "functional groups fingerprint Tanimoto index (median)","functional groups fingerprint Tanimoto index (percentile 80)",
        "functional groups fingerprint Tanimoto index (percentile 90)","functional groups fingerprint Tanimoto index (percentile 95)",      
      ]
file_names=["MorganR2average","MorganR2max","MorganR2median","MorganR2perc80","MorganR2perc90","MorganR2perc95",
            "MorganR3average","MorganR3max","MorganR3median","MorganR3perc80","MorganR3perc90","MorganR3perc95",
            "MorganR4average","MorganR4max","MorganR4median","MorganR4perc80","MorganR4perc90","MorganR4perc95",
            "functgrpsaverage","functgrpsmax","functgrpsmedian","functgrpsperc80","functgrpsperc90","functgrpsperc95",
            ]
scales=[[0.0,0.3],[0,1.2],[0,0.3],[0,0.3],[0,0.8],[0,1.2]]*4
scales=[[0.0,s] for s in [0.22,1.1,0.22,0.34,0.34,0.34,0.16,1.1,0.16,0.24,0.28,0.28,0.16,1.1,0.16,0.22,0.26,0.26,0.42,1.1,0.42,0.22,0.26,0.26]]

legend_position=[[0.4,0.95]]*len(props)
for prop,name_of_property,scale,legend_position,file_name in zip(props,names_of_prop,scales,legend_position,file_names):
    MW_histograms=[]
    scatters=[]
    for db,color,name in zip([GPKA_db,SAMPL_db,CHEMBL_with_pka_db,PUBCHEM_with_pka_db],
                             ["red","orange","green","blue"],
                             ["GpKa","SAMPL","CHEMBL(pKa)","PUBCHEM(pKa)"]):
            #MW_histograms.append(go.Histogram(x=db["MW"],opacity=0.75,xbins={"size":0.10},marker_color=color,legendgroup=1,legendrank=2))
            histogram=go.Histogram(x=list(db[prop]),opacity=1.0,marker_color=color,histnorm='percent',name=name,
                                              autobinx=False,#nbinsx=20
                                                 xbins=dict(start=scale[0],end=scale[1], size=(scale[1]-scale[0])/20))

            xx=np.linspace(histogram.xbins['start'],histogram.xbins['end']+histogram.xbins['size'],100)
            try: 
                density = scipy.stats.gaussian_kde(db[prop])
                yy=density(xx)
                #calculate the scale
                plotbins = list(np.arange(start=histogram.xbins['start'], stop=histogram.xbins['end']+histogram.xbins['size'], step=histogram.xbins['size']))
                counts, bins = np.histogram(db[prop], bins=plotbins)
                scale_factor=100*np.max(counts)/(np.max(yy)*np.sum(counts))
                yy=yy*scale_factor
            except: yy=np.zeros(len(xx))
            sct=go.Scatter(x=list(xx),y=list(yy),mode='lines',showlegend=False,hoverinfo="skip",
                           line_color=color,
                           #marker=dict(color=color, line=dict(width=1),showscale=False),
                           fill='tozeroy',
                          )
            #go.Figure(sct).show()
            MW_histograms.append(histogram)
            scatters.append(sct)
    
        
    fig=go.Figure(data=MW_histograms[0])
    fig.add_trace(scatters[0])
    fig.add_trace(MW_histograms[1])
    fig.add_trace(scatters[1])
    fig.add_trace(MW_histograms[2])
    fig.add_trace(scatters[2])
    fig.add_trace(MW_histograms[3])
    fig.add_trace(scatters[3])
       
    fig.update_layout(height=800,xaxis_range=scale,#yaxis_range=[0,6],
                          legend={"yanchor":"top","xanchor":"right","y":legend_position[1],"x":legend_position[0],"font":{"size":34}},
                      bargap=0.6,bargroupgap=0.0)
    fig.update_xaxes(title_text=name_of_property,title_font={'size': 32, 'weight': 1000},tickfont={"size":28})
    fig.update_yaxes(title_text="%",title_font={'size': 36, 'weight': 1000},tickfont={"size":24})
        
    #fig.show()
    fig.write_html(file_name+".html")
    fig.write_image(file_name+".png", width=1200, height=800,scale=1)


In [11]:
CHEMBL_with_pka_db=pd.read_csv("CHEMBL_with_pka_database_with_descriptors_tversky_with_Gpka.csv",encoding='unicode_escape')
#CH,line_shape='spline'EMBL_with_pka_small_db= CHEMBL_with_pka_db[CHEMBL_with_pka_db["MW"]<300]
PUBCHEM_with_pka_db=pd.read_csv("PubChem_with_pka_database_with_descriptors_tversky_with_Gpka.csv",encoding='unicode_escape')
GPKA_db=pd.read_csv("Gpka_database_with_descriptors_tversky_with_Gpka.csv")
SAMPL_db=pd.read_csv("SAMPL_database_with_descriptors_tversky_with_Gpka.csv")

import scipy.stats
    
for db in [CHEMBL_with_pka_db,SAMPL_db,PUBCHEM_with_pka_db,GPKA_db]: 
    db.set_index("name",inplace=True)
    db.dropna(how='all', axis=1, inplace=True)

MW_histograms=[]
props=["Morgan radius 2 tversky similarity average","Morgan radius 2 tversky similarity max",
      "Morgan radius 2 tversky similarity median","Morgan radius 2 tversky similarity percentile 80",
      "Morgan radius 2 tversky similarity percentile 90","Morgan radius 2 tversky similarity percentile 90",
        "Morgan radius 3 tversky similarity average","Morgan radius 3 tversky similarity max",
      "Morgan radius 3 tversky similarity median","Morgan radius 3 tversky similarity percentile 80",
      "Morgan radius 3 tversky similarity percentile 90","Morgan radius 3 tversky similarity percentile 90",      
        "Morgan radius 4 tversky similarity average","Morgan radius 4 tversky similarity max",
      "Morgan radius 4 tversky similarity median","Morgan radius 4 tversky similarity percentile 80",
      "Morgan radius 4 tversky similarity percentile 90","Morgan radius 4 tversky similarity percentile 90", 
     "custom fingerprint tversky similarity average","custom fingerprint tversky similarity max",
      "custom fingerprint tversky similarity median","custom fingerprint tversky similarity percentile 80",
      "custom fingerprint tversky similarity percentile 90","custom fingerprint tversky similarity percentile 90",      
      ]
names_of_prop=["Morgan (R=2) fingerprint Tversky(0,1) index (average)","Morgan (R=2) fingerprint Tversky(0,1) index (max)",
      "Morgan (R=2) fingerprint Tversky(0,1) index (median)","Morgan (R=2) fingerprint Tversky(0,1) index (percentile 80)",
        "Morgan (R=2) fingerprint Tversky(0,1) index (percentile 90)","Morgan (R=2) fingerprint Tversky(0,1) index (percentile 95)",
       "Morgan (R=3) fingerprint Tversky(0,1) index (average)","Morgan (R=3) fingerprint Tversky(0,1) index (max)",
      "Morgan (R=3) fingerprint Tversky(0,1) index (median)","Morgan (R=3) fingerprint Tversky(0,1) index (percentile 80)",
        "Morgan (R=3) fingerprint Tversky(0,1) index (percentile 90)","Morgan (R=3) fingerprint Tversky(0,1) index (percentile 95)",
       "Morgan (R=4) fingerprint Tversky(0,1) index (average)","Morgan (R=4) fingerprint Tversky(0,1) index (max)",
      "Morgan (R=4) fingerprint Tversky(0,1) index (median)","Morgan (R=4) fingerprint Tversky(0,1) index (percentile 80)",
        "Morgan (R=4) fingerprint Tversky(0,1) index (percentile 90)","Morgan (R=4) fingerprint Tversky(0,1) index (percentile 95)",
       "functional groups fingerprint Tversky(0,1) index (average)","functional groups fingerprint Tversky(0,1) index (max)",
      "functional groups fingerprint Tversky(0,1) index (median)","functional groups fingerprint Tversky(0,1) index (percentile 80)",
        "functional groups fingerprint Tversky(0,1) index (percentile 90)","functional groups fingerprint Tversky(0,1) index (percentile 95)",      
      ]
file_names=["MorganR2tverskyaverage","MorganR2tverskymax","MorganR2tverskymedian","MorganR2tverskyperc80","MorganR2tverskyperc90","MorganR2tverskyperc95",
            "MorganR3tverskyaverage","MorganR3tverskymax","MorganR3tverskymedian","MorganR3tverskyperc80","MorganR3tverskyperc90","MorganR3tverskyperc95",
            "MorganR4tverskyaverage","MorganR4tverskymax","MorganR4tverskymedian","MorganR4tverskyperc80","MorganR4tverskyperc90","MorganR4tverskyperc95",
            "functgrp_tverskysaverage","functgrp_tverskysmax","functgrp_tverskysmedian","functgrp_tverskysperc80","functgrp_tverskysperc90","functgrp_tverskysperc95",
            ]
scales=[[0.0,0.3],[0,1.2],[0,0.3],[0,0.3],[0,0.8],[0,1.2]]*4
scales=[[0.0,s] for s in [0.45,1.1,0.45,0.6,0.65,0.65,0.4,1.1,0.4,0.5,0.6,0.6,0.4,1.1,0.4,0.45,0.5,0.5,1.0,1.0,1.0,0.5,0.6,0.6]]
#scales=[[0,1]]*24
legend_position=[[0.4,0.95]]*len(props)
for prop,name_of_property,scale,legend_position,file_name in zip(props,names_of_prop,scales,legend_position,file_names):
    MW_histograms=[]
    scatters=[]
    for db,color,name in zip([GPKA_db,SAMPL_db,CHEMBL_with_pka_db,PUBCHEM_with_pka_db],
                             ["red","orange","green","blue"],
                             ["GpKa","SAMPL","CHEMBL(pKa)","PUBCHEM(pKa)"]):
            #MW_histograms.append(go.Histogram(x=db["MW"],opacity=0.75,xbins={"size":0.10},marker_color=color,legendgroup=1,legendrank=2))
#
            
            histogram=go.Histogram(x=list(db[prop]),opacity=1.0,marker_color=color,histnorm='percent',name=name,
                                              autobinx=False,#nbinsx=20
                                                 xbins=dict(start=scale[0],end=scale[1], size=(scale[1]-scale[0])/20))

            xx=np.linspace(histogram.xbins['start'],histogram.xbins['end']+histogram.xbins['size'],100)
            try: 
                density = scipy.stats.gaussian_kde(db[prop])
                yy=density(xx)
                #calculate the scale
                plotbins = list(np.arange(start=histogram.xbins['start'], stop=histogram.xbins['end']+histogram.xbins['size'], step=histogram.xbins['size']))
                counts, bins = np.histogram(db[prop], bins=plotbins)
                scale_factor=100*np.max(counts)/(np.max(yy)*np.sum(counts))
                yy=yy*scale_factor
            except: yy=np.zeros(len(xx))
            sct=go.Scatter(x=list(xx),y=list(yy),mode='lines',showlegend=False,hoverinfo="skip",
                           line_color=color,
                           #marker=dict(color=color, line=dict(width=1),showscale=False),
                           fill='tozeroy',
                          )
            #go.Figure(sct).show()
            MW_histograms.append(histogram)
            scatters.append(sct)
                                      
    #kde=ff.create_distplot(distplot_data,names_of_prop,show_rug=False)
    #kde.show()            
        
    fig=go.Figure(data=MW_histograms[0])
    fig.add_trace(scatters[0])
    fig.add_trace(MW_histograms[1])
    fig.add_trace(scatters[1])
    fig.add_trace(MW_histograms[2])
    fig.add_trace(scatters[2])
    fig.add_trace(MW_histograms[3])
    fig.add_trace(scatters[3])
       
    fig.update_layout(height=800,xaxis_range=scale,#yaxis_range=[0,6],
                          legend={"yanchor":"top","xanchor":"right","y":legend_position[1],"x":legend_position[0],"font":{"size":34}},
                      bargap=0.6,bargroupgap=0.0)
    fig.update_xaxes(title_text=name_of_property,title_font={'size': 32, 'weight': 1000},tickfont={"size":28})
    fig.update_yaxes(title_text="%",title_font={'size': 36, 'weight': 1000},tickfont={"size":24})
        
    #fig.show()
    fig.write_html(file_name+".html")
    fig.write_image(file_name+".png", width=1200, height=800,scale=1)


In [12]:
from routes import extracted_data_route
from drop_compounds import drop_compounds
import copy
lot="swb97xd"
csv_file=extracted_data_route+"values_extracted-gibbs-"+lot+".25.csv" 
data=pd.read_csv(csv_file,low_memory=True)
data.dropna(axis=0)

outliers=data[data["compn"].isin( drop_compounds)]
data=data[~data["compn"].isin(drop_compounds)]

#data.set_index("compn",inplace=True)
#outliers.set_index("compn",inplace=True)

from sklearn.linear_model import LinearRegression
test_attributes='deltaG'
X=np.c_[data['deltaG']]
Y=np.array(data["pKa"].copy())
lin_reg = LinearRegression()
lin_reg.fit(X,Y)
pka_prediction=lin_reg.predict(X)


fig3=go.Figure(data=go.Scatter(
                               y=list(pka_prediction),x=list(Y),mode='markers',showlegend=False,hoverinfo="skip",
                                marker=dict(color="lightgray", line=dict(width=1),showscale=False) 
                              )) 


outliers_X,outliers_Y= np.c_[outliers[test_attributes]],np.array(outliers["pKa"].copy())
outliers_pka_prediction=lin_reg.predict(outliers_X)
fig3.add_trace(go.Scatter( 
                            y=list(outliers_pka_prediction),x=list(outliers_Y),mode='markers',text=outliers["compn"],showlegend=False,
                                marker=dict(color="red", line=dict(width=2),showscale=False) 
                              )) 
fig3.update_layout(height=1100,width=1100,#xaxis_range=x_scale,#yaxis_range=[0,6],
                  legend={"yanchor":"top","xanchor":"right","y":0.99,"x":0.9,"font":{"size":22}})
fig3.update_xaxes(title_text="published pKa",title_font={'size': 22, 'weight': 1000},tickfont={"size":16})
fig3.update_yaxes(title_text="predicted pKa",title_font={'size': 22, 'weight': 1000},tickfont={"size":16})
#fig3.show()
fig3.write_html("dropped_compounds.html")
fig3.write_image("dropped_compounds.png", width=1600, height=1600,scale=4)
